# exp05 — HITL part-panel segmentation, SPLIT classes (fender/quarter, front/back door, front/back wheel, windshield/back-windshield)

**Purpose (one variable under test):** same dataset, same recipe as exp04
(`exp04_hitl_panels.ipynb`, untouched by this notebook) — the ONLY change is
`HITL_TO_CANONICAL`/`CANONICAL_CLASSES` in Section 5, which stops merging four pairs
of raw HITL classes that exp04 collapsed:

| exp04 (merged) | exp05 (split into) |
|---|---|
| `fender` (Fender + Quarter-panel) | `fender`, `quarter_panel` |
| `door` (Front-door + Back-door) | `front_door`, `back_door` |
| `wheel` (Front-wheel + Back-wheel) | `front_wheel`, `back_wheel` |
| `windshield` (Windshield + Back-windshield) | `windshield`, `back_windshield` |

16 canonical classes → **20** (10 costed panels, up from 8; 10 non-panel trained-only,
up from 8 — `window`, `headlight`, `taillight`, `mirror`, `grille`, `license_plate`
unchanged, `wheel` is replaced by its two split halves, `windshield` keeps its name for
one half and gains a sibling for the other).

**Why split, not re-merge-if-bad:** a split can be collapsed back to the coarser label
at *inference* time with a pure remap (no retraining) if a half turns out too weak —
see `run_offline.py`'s new `MERGE_REAR_WINDSHIELD` switch for exactly this fallback on
`back_windshield`. A merge can't be undone without retraining. So split now, decide
per-class later from the numbers this run produces.

**Known risk going in (flagged before training, not after):** raw HITL instance counts
(pulled from exp04's own `hitl_instance_counts_raw.csv` on Drive, pre-mapping) —
`fender` 935, `front_door` 917, `front_wheel` 885, `back_wheel` 865, `quarter_panel` 855,
`back_door` 721, `windshield` 632, **`back_windshield` 271**. exp04's two weakest
*already-trained* classes were `trunk` (416 instances → P 0.73 / R 0.67) and
`license_plate` (370 → P 0.79 / R 0.67). `back_windshield` at 271 is below both of those,
with only ~27 test images to evaluate it on — Section 8a's new back-windshield cell
reports this explicitly as low-confidence (n), and Section 8a's new combined-glass cell
checks that splitting it didn't regress overall glass detection vs. exp04's merged
`windshield` baseline. If either looks bad, the `MERGE_REAR_WINDSHIELD` runner switch
is the documented way back without retraining — see that file.

**Design constraints (unchanged from exp04):**
- One dataset (HITL), one training run — no sweep. Same model source, imgsz, epochs,
  patience, weight_decay, batch, seed, augmentation as exp04 (Section 7 below).
- Every stage reads its input from Drive and writes its output to Drive. Any cell can
  be re-run independently.
- Do not edit the backend, Flutter, or the labor-hours table.
- `exp04_hitl_panels.ipynb` and its Drive outputs are **not modified** by this notebook
  — exp05 writes to its own `EXP_DIR` (Section 0) and only *reads* exp04's saved CSVs
  for side-by-side comparison in the new Section 8a cells.

**New in exp05 vs. exp04 (all additions, no removals from the original pipeline):**
1. Section 5: the four-pair split mapping above (only functional change).
2. Section 8a: back-windshield low-n report, combined-glass-vs-exp04 check, and a
   split-pair confusion/accuracy summary (fender↔quarter_panel, front_door↔back_door,
   front_wheel↔back_wheel) side by side with exp04's merged numbers.
3. Section 8b: a ground-truth compatibility check — the shared `real_eval_ground_truth_panels.csv`
   was labeled against exp04's 8 panel classes, so any image whose true part is now a
   *specific* half of a split pair needs its `true_panel` value re-checked before this
   run's panel accuracy is comparable to exp04's.
4. Section 8c (new): scores the attached `panel_ground_truth_31photos.csv` — for the 26
   rows flagged `tests_split=yes`, checks whether the damage region's IoA≥0.40-associated
   part gets the correct **split** label, with an accuracy number and a misses list.

**Assumptions (same as exp04 — revisit if Section 4 contradicts them):**
1. The HITL Kaggle export ships polygon instance annotations (COCO or VIA/VGG,
   auto-detected in Section 4).
2. HITL's 21 raw part classes still carry no left/right signal — only front/back, which
   is exactly the signal this experiment now preserves for four of those pairs instead
   of discarding it.
3. Initialization checkpoint: `yolov8s-seg.pt`, same as exp04 (not any carparts-seg
   lineage checkpoint).
4. No automated ground truth exists for the 20 real CrashLens close-ups or the 31-photo
   Tagdeer set — both use shared, manually-labeled CSVs on Drive, same honesty rule as
   exp04.


## 📁 Section 1 — Setup: mount Drive, install deps
Independently re-runnable. No dependency on anything having survived a previous session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HUB_DISABLE_XET'] = '1'  # avoids a flaky xet handshake seen in earlier experiments; harmless if unused here

!pip install -q ultralytics kaggle

import ultralytics
ultralytics.checks()


In [ ]:
import os

# ---- Fixed paths (all under Drive; nothing here depends on /content surviving) ----
DRIVE_ROOT   = '/content/drive/MyDrive/carparts/segmentation'
EXP_ID       = 'exp05_hitl_panels_split'
EXP_DIR      = os.path.join(DRIVE_ROOT, 'experiments', EXP_ID)
DATASET_VERSION = 'v4_hitl_split'   # new version: re-converted labels under the split mapping (Section 5)
DATASET_DIR  = os.path.join(DRIVE_ROOT, 'datasets', DATASET_VERSION)

# exp04's experiment dir -- READ ONLY, for side-by-side comparison in Section 8a. Never
# written to by this notebook.
EXP04_DIR = os.path.join(DRIVE_ROOT, 'experiments', 'exp04_hitl_panels')

# Real CrashLens close-ups used for the final reality check (Section 8b) — same
# location exp01/exp02/exp03/exp04 already read from.
REAL_DATA_ROOT = '/content/drive/MyDrive/carparts'
REAL_CROPPED_DIR = os.path.join(REAL_DATA_ROOT, 'cropped')

# Shared, cross-experiment manual ground-truth file for the 20 real close-ups (panel
# labels only). Lives one level above any single experiment so exp05+ can reuse it --
# labeled against exp04's 8 panel classes, see Section 8b's new compatibility check.
REAL_GT_PATH = os.path.join(DRIVE_ROOT, 'real_eval_ground_truth_panels.csv')

# Section 8c (new): the 31-photo Tagdeer association-accuracy check. Upload the CSV
# (and the 31 images, if not already on Drive under REAL_DATA_ROOT) before running
# Section 8c -- see that section's markdown for the expected schema.
PANEL_GT_31PHOTOS_PATH = os.path.join(DRIVE_ROOT, 'panel_ground_truth_31photos.csv')
PHOTOS_31_DIR = os.path.join(REAL_DATA_ROOT, 'tagdeer_31photos')  # adjust if uploaded elsewhere

for sub in ['weights', 'val_preds', 'label_overlay_check', 'confusion_matrix']:
    os.makedirs(os.path.join(EXP_DIR, sub), exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

SEED = 42

print('EXP_DIR    :', EXP_DIR)
print('EXP04_DIR  :', EXP04_DIR, '(exists)' if os.path.isdir(EXP04_DIR) else '(NOT FOUND -- Section 8a comparisons will skip)')
print('DATASET_DIR:', DATASET_DIR)


## 🔑 Section 2 — Kaggle authentication

Reads the Colab secret `KAGGLE_API_TOKEN`. It may hold either the **full contents of
`kaggle.json`** (`{"username": ..., "key": ...}`) or just the **raw API key** — in the
latter case a second secret `KAGGLE_USERNAME` is also required. No manual steps beyond
adding the secret(s) in the Colab secrets panel (key icon, left sidebar).

In [ ]:
import json
import os
from google.colab import userdata

token_raw = userdata.get('KAGGLE_API_TOKEN')
if not token_raw:
    raise ValueError(
        "Colab secret 'KAGGLE_API_TOKEN' not found. Add it via the secrets panel "
        "(key icon in the left sidebar) before running this cell."
    )
token_raw = token_raw.strip()

kaggle_json = None
try:
    parsed = json.loads(token_raw)
    if isinstance(parsed, dict) and 'username' in parsed and 'key' in parsed:
        kaggle_json = {'username': parsed['username'], 'key': parsed['key']}
except json.JSONDecodeError:
    pass

if kaggle_json is None:
    # KAGGLE_API_TOKEN held a raw key, not a full kaggle.json — need the username too.
    username = userdata.get('KAGGLE_USERNAME')
    if not username:
        raise ValueError(
            "'KAGGLE_API_TOKEN' looks like a raw key, not a full kaggle.json. "
            "Add a 'KAGGLE_USERNAME' Colab secret as well."
        )
    kaggle_json = {'username': username.strip(), 'key': token_raw}

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(kaggle_json_path, 'w') as f:
    json.dump(kaggle_json, f)
os.chmod(kaggle_json_path, 0o600)

print(f"kaggle.json written for user '{kaggle_json['username']}'")


## ⬇️ Section 3 — Acquire HITL dataset from Kaggle

Reproducible download (not a local upload) into `DATASET_DIR/raw` on **Drive**. Skips
automatically if `raw/` already exists and is non-empty, so re-running this cell after
a disconnect is a no-op rather than a re-download.

In [ ]:
import subprocess
from pathlib import Path

RAW_DIR = os.path.join(DATASET_DIR, 'raw')

def _dir_has_files(p):
    return os.path.isdir(p) and any(f.is_file() for f in Path(p).rglob('*'))

if _dir_has_files(RAW_DIR):
    n_files = sum(1 for f in Path(RAW_DIR).rglob('*') if f.is_file())
    print(f'RAW already present at {RAW_DIR} ({n_files} files) — skipping download.')
else:
    os.makedirs(RAW_DIR, exist_ok=True)
    local_zip_dir = '/content/hitl_download'
    os.makedirs(local_zip_dir, exist_ok=True)

    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'humansintheloop/car-parts-and-car-damages',
         '-p', local_zip_dir],
        check=True,
    )

    zips = list(Path(local_zip_dir).glob('*.zip'))
    if not zips:
        raise RuntimeError(f'Kaggle download reported success but no .zip found in {local_zip_dir}')
    zip_path = zips[0]

    import zipfile
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(RAW_DIR)

    n_files = sum(1 for f in Path(RAW_DIR).rglob('*') if f.is_file())
    print(f'Extracted {n_files} files to {RAW_DIR}')


## 🔎 Section 4 — Inspection (report only — does not stop the run)

Parses whatever polygon-annotation format the HITL export actually uses, and separates
the **car-parts** annotations from the **car-damages** annotations (same dataset ships
both; only parts annotations are relevant here). Prints + saves a per-class instance
count table and a 12-image contact sheet so framing (whole-car vs close-up) is visible
before any conversion happens. This informs interpretation of later results — it is not
a gate.

In [ ]:
import json
from pathlib import Path
from collections import Counter

# Raw HITL part-class name hints, used only to tell "car parts" annotation files apart
# from "car damages" ones (different category vocab: scratch/dent/crack/...).
_HITL_PART_HINTS = {
    'door', 'bumper', 'fender', 'hood', 'trunk', 'roof', 'rocker', 'quarter',
    'windshield', 'window', 'headlight', 'tail-light', 'taillight', 'wheel',
    'mirror', 'grille', 'license',
}

def _category_overlap_ratio(cat_names):
    if not cat_names:
        return 0.0
    norm = [c.strip().lower() for c in cat_names]
    hits = sum(1 for c in norm if any(h in c for h in _HITL_PART_HINTS))
    return hits / len(norm)


def _try_parse_coco(json_path):
    """Returns (records, category_names) or None if this file isn't COCO-shaped."""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return None
    if not isinstance(data, dict) or not all(k in data for k in ('images', 'annotations', 'categories')):
        return None

    images = {im['id']: im for im in data['images']}
    cats = {c['id']: c['name'] for c in data['categories']}
    records = {
        img_id: {'file_name': im['file_name'], 'width': im.get('width'), 'height': im.get('height'), 'instances': []}
        for img_id, im in images.items()
    }
    for ann in data['annotations']:
        seg = ann.get('segmentation')
        img_id = ann.get('image_id')
        if img_id not in records or not seg:
            continue
        # polygon form only (list-of-lists of coords); skip RLE masks
        if not (isinstance(seg, list) and seg and isinstance(seg[0], list)):
            continue
        records[img_id]['instances'].append({
            'category': cats.get(ann['category_id'], f"id_{ann['category_id']}"),
            'polygons': seg,
        })
    return records, list(cats.values())


def _try_parse_via(json_path):
    """Returns (records, category_names) or None if this file isn't VIA-shaped."""
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except (json.JSONDecodeError, UnicodeDecodeError):
        return None
    meta = data.get('_via_img_metadata', data) if isinstance(data, dict) else None
    if not isinstance(meta, dict) or not meta:
        return None
    sample = next(iter(meta.values()))
    if not isinstance(sample, dict) or 'regions' not in sample:
        return None

    records, cat_names = {}, []
    for idx, (key, entry) in enumerate(meta.items()):
        fname = entry.get('filename', key)
        instances = []
        for region in entry.get('regions', []):
            shape = region.get('shape_attributes', {})
            xs, ys = shape.get('all_points_x'), shape.get('all_points_y')
            if not xs or not ys:
                continue
            poly = [v for pair in zip(xs, ys) for v in pair]
            cat = None
            for attr_val in region.get('region_attributes', {}).values():
                if isinstance(attr_val, str) and attr_val:
                    cat = attr_val
                    break
                if isinstance(attr_val, dict):
                    trues = [k for k, v in attr_val.items() if v]
                    if trues:
                        cat = trues[0]
                        break
            cat = cat or 'unknown'
            cat_names.append(cat)
            instances.append({'category': cat, 'polygons': [poly]})
        records[idx] = {'file_name': fname, 'width': None, 'height': None, 'instances': instances}
    return records, cat_names


def discover_part_annotation_files(raw_dir):
    """Scans raw_dir for *.json, parses each as COCO or VIA, and returns the
    parsed file(s) whose category vocabulary looks like car PARTS (not damages)."""
    candidates = []
    for jp in sorted(Path(raw_dir).rglob('*.json')):
        parsed = _try_parse_coco(jp)
        fmt = 'coco'
        if parsed is None:
            parsed = _try_parse_via(jp)
            fmt = 'via'
        if parsed is None:
            continue
        records, cat_names = parsed
        ratio = _category_overlap_ratio(cat_names)
        print(f'  {jp.relative_to(raw_dir)}  [{fmt}]  {len(records)} images, '
              f'{len(set(cat_names))} categories, part-hint overlap={ratio:.2f}')
        candidates.append((jp, fmt, records, cat_names, ratio))
    return candidates


print('Scanning annotation files under', RAW_DIR)
_candidates = discover_part_annotation_files(RAW_DIR)
if not _candidates:
    raise RuntimeError(
        f'No parseable COCO- or VIA-style JSON found under {RAW_DIR}. '
        'Inspect the raw export by hand and adjust discover_part_annotation_files().'
    )

PART_ANNOTATION_FILES = [c for c in _candidates if c[4] >= 0.5]
if not PART_ANNOTATION_FILES:
    print('⚠️ No file cleared the 0.5 part-hint-overlap threshold. Falling back to the '
          'single best-scoring file — verify this is actually the parts annotation, not damages.')
    PART_ANNOTATION_FILES = [max(_candidates, key=lambda c: c[4])]

print()
print('Selected as PART annotations:')
for jp, fmt, records, cat_names, ratio in PART_ANNOTATION_FILES:
    print(f'  {jp.name}  [{fmt}]  overlap={ratio:.2f}  images={len(records)}')


In [ ]:
# Build a filename -> absolute path index of every image file under RAW_DIR, since the
# JSON's file_name field may not match the extracted directory layout exactly.
_IMAGE_EXTS = {'.jpg', '.jpeg', '.png'}
_image_index = {}
for f in Path(RAW_DIR).rglob('*'):
    if f.is_file() and f.suffix.lower() in _IMAGE_EXTS:
        _image_index.setdefault(f.name, f)

def resolve_image_path(file_name):
    return _image_index.get(Path(file_name).name)


# Merge every selected part-annotation file into one unified pool, keyed by resolved
# absolute image path (skips instances whose image can't be located on disk).
HITL_RECORDS = {}  # abs_path (str) -> {'width', 'height', 'instances': [...]}
_unresolved = 0
for jp, fmt, records, cat_names, ratio in PART_ANNOTATION_FILES:
    for rec in records.values():
        img_path = resolve_image_path(rec['file_name'])
        if img_path is None:
            _unresolved += 1
            continue
        key = str(img_path)
        if key not in HITL_RECORDS:
            HITL_RECORDS[key] = {'width': rec.get('width'), 'height': rec.get('height'), 'instances': []}
        HITL_RECORDS[key]['instances'].extend(rec['instances'])

print(f'Merged {len(HITL_RECORDS)} images with resolvable files ({_unresolved} instances skipped — image not found on disk).')


In [ ]:
# Per-class instance count table (raw HITL class names, pre-mapping)
import pandas as pd

_raw_counts = Counter()
for rec in HITL_RECORDS.values():
    for inst in rec['instances']:
        _raw_counts[inst['category']] += 1

counts_df = pd.DataFrame(sorted(_raw_counts.items(), key=lambda kv: -kv[1]), columns=['hitl_class', 'instance_count'])
print(counts_df.to_string(index=False))

counts_csv_path = os.path.join(EXP_DIR, 'hitl_instance_counts_raw.csv')
counts_df.to_csv(counts_csv_path, index=False)
print('\nSaved:', counts_csv_path)


In [ ]:
# 12-image contact sheet — polygon overlays, to eyeball framing (whole-car vs close-up)
# before any conversion. Reused again in Section 8b for the real CrashLens predictions.
import random
import cv2
import numpy as np
import matplotlib.pyplot as plt

def draw_polygons(image_bgr, instances, color=(0, 255, 0)):
    out = image_bgr.copy()
    for inst in instances:
        for poly in inst['polygons']:
            pts = np.array(poly, dtype=np.float32).reshape(-1, 2).astype(np.int32)
            if len(pts) < 3:
                continue
            cv2.polylines(out, [pts], isClosed=True, color=color, thickness=2)
        if inst['polygons']:
            x, y = np.array(inst['polygons'][0][:2], dtype=np.int32)
            cv2.putText(out, inst['category'], (x, max(y, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    return out


def save_contact_sheet(panels, out_path, ncols=4, figsize_per_cell=3.2, suptitle=None):
    """panels: list of (rgb_image_array, caption) pairs."""
    n = len(panels)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per_cell * ncols, figsize_per_cell * nrows))
    axes = np.atleast_2d(axes).reshape(nrows, ncols)
    for i in range(nrows * ncols):
        ax = axes[i // ncols][i % ncols]
        ax.axis('off')
        if i < n:
            img, caption = panels[i]
            ax.imshow(img)
            ax.set_title(caption, fontsize=8)
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    fig.savefig(out_path, dpi=120)
    plt.close(fig)


_sample_keys = random.Random(SEED).sample(list(HITL_RECORDS.keys()), k=min(12, len(HITL_RECORDS)))
_panels = []
for key in _sample_keys:
    rec = HITL_RECORDS[key]
    img_bgr = cv2.imread(key)
    if img_bgr is None:
        continue
    overlay = draw_polygons(img_bgr, rec['instances'])
    overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
    n_inst = len(rec['instances'])
    _panels.append((overlay_rgb, f'{Path(key).name}\n({n_inst} instances)'))

contact_sheet_path = os.path.join(EXP_DIR, 'contact_sheet_hitl_raw.png')
save_contact_sheet(_panels, contact_sheet_path, suptitle='HITL raw sample (framing check)')
print('Saved:', contact_sheet_path)


## 🗂️ Section 5 — Class map + convert to YOLO-seg

Maps **all 21 HITL part classes** to **20 canonical classes** — 10 costed **panels**
and 10 non-panel classes that are trained (so the model has a correct bucket for them)
but never costed by this model. The ONLY change from exp04: four pairs that exp04
merged are now kept separate (front/back is a real, present signal in the raw HITL
labels for these four — see Section 0's assumption 2):

| HITL class | canonical | costed? |
|---|---|---|
| Front-door | `front_door` | panel |
| Back-door | `back_door` | panel |
| Front-bumper | `front_bumper` | panel |
| Back-bumper | `back_bumper` | panel |
| Fender | `fender` | panel |
| Quarter-panel | `quarter_panel` | panel |
| Hood | `hood` | panel |
| Trunk | `trunk` | panel |
| Roof | `roof` | panel |
| Rocker-panel | `sill` | panel |
| Windshield | `windshield` | — (glass hours come from damage type) |
| Back-windshield | `back_windshield` | — (⚠️ 271 raw instances, ~27 test images -- see Section 0's low-n flag and Section 8a's dedicated report) |
| Front-window, Back-window | `window` | — (NOT split -- out of scope for this round, see Section 0) |
| Headlight | `headlight` | — (lamp hours come from damage type) |
| Tail-light | `taillight` | — |
| Front-wheel | `front_wheel` | — (tire hours come from damage type) |
| Back-wheel | `back_wheel` | — |
| Mirror | `mirror` | — (not costed today) |
| Grille | `grille` | — |
| License-plate | `license_plate` | — |

No HITL class is dropped. Front/back is now PRESERVED for door, fender/quarter-panel,
wheel, and windshield (exp04 merged all four); it remains merged only for window
(front-window/back-window → `window`), which this round does not touch.

In [ ]:
CANONICAL_CLASSES = [
    # --- panels: costed, judged in Section 8 -- 10, up from exp04's 8 ---
    'front_door', 'back_door', 'front_bumper', 'back_bumper',
    'fender', 'quarter_panel', 'hood', 'trunk', 'roof', 'sill',
    # --- non-panel: trained so the model has a correct bucket, never costed here -- 10 ---
    'windshield', 'back_windshield', 'window', 'headlight', 'taillight',
    'front_wheel', 'back_wheel', 'mirror', 'grille', 'license_plate',
]
PANEL_CLASSES = set(CANONICAL_CLASSES[:10])
CANONICAL_INDEX = {name: i for i, name in enumerate(CANONICAL_CLASSES)}

assert len(CANONICAL_CLASSES) == 20 and len(PANEL_CLASSES) == 10

# The four split pairs this experiment is actually testing (vs. exp04's merged halves) --
# used by Section 8a's new split-pair comparison cells, kept here so Section 5 stays the
# single source of truth for "what changed."
SPLIT_PAIRS = {
    'fender': ('fender', 'quarter_panel'),
    'door': ('front_door', 'back_door'),
    'wheel': ('front_wheel', 'back_wheel'),
    'windshield': ('windshield', 'back_windshield'),
}

HITL_TO_CANONICAL = {
    'front-door': 'front_door', 'back-door': 'back_door',
    'front-bumper': 'front_bumper', 'back-bumper': 'back_bumper',
    'fender': 'fender', 'quarter-panel': 'quarter_panel',
    'hood': 'hood', 'trunk': 'trunk', 'roof': 'roof', 'rocker-panel': 'sill',
    'windshield': 'windshield', 'back-windshield': 'back_windshield',
    'front-window': 'window', 'back-window': 'window',
    'headlight': 'headlight', 'tail-light': 'taillight',
    'front-wheel': 'front_wheel', 'back-wheel': 'back_wheel',
    'mirror': 'mirror', 'grille': 'grille', 'license-plate': 'license_plate',
}

def normalize_hitl_name(name):
    return name.strip().lower().replace(' ', '-').replace('_', '-')

def map_to_canonical(hitl_name):
    return HITL_TO_CANONICAL.get(normalize_hitl_name(hitl_name))


# Coverage check against whatever categories Section 4 actually found — fail loudly
# rather than silently dropping an unrecognized HITL class.
_present = {inst['category'] for rec in HITL_RECORDS.values() for inst in rec['instances']}
_unmapped = sorted(c for c in _present if map_to_canonical(c) is None)
if _unmapped:
    raise RuntimeError(
        f'HITL_TO_CANONICAL does not cover these raw categories found in the data: {_unmapped}. '
        'Add them to the mapping table above before converting (do not silently drop parts).'
    )
print(f'All {len(_present)} raw HITL categories are covered by the mapping.')

# Log the full mapping table to notes-facing text now (Section 9 will fold this in).
_mapping_lines = ['HITL raw class -> canonical class (costed?)']
for raw, canon in sorted(HITL_TO_CANONICAL.items()):
    _mapping_lines.append(f'  {raw:16s} -> {canon:16s} ({"panel" if canon in PANEL_CLASSES else "non-panel"})')
CLASS_MAP_LOG = '\n'.join(_mapping_lines)
print(CLASS_MAP_LOG)


In [ ]:
# Convert every HITL instance into a single YOLO-seg label line, using the largest
# polygon part when an instance has multiple (e.g. occlusion splits), and copy the
# source image alongside its label into a flat "converted" pool on Drive. Splitting
# into train/val/test happens in Section 6 — this section only builds the pool.
import shutil
import hashlib

CONVERTED_IMAGES_DIR = os.path.join(DATASET_DIR, 'converted', 'images')
CONVERTED_LABELS_DIR = os.path.join(DATASET_DIR, 'converted', 'labels')
os.makedirs(CONVERTED_IMAGES_DIR, exist_ok=True)
os.makedirs(CONVERTED_LABELS_DIR, exist_ok=True)

def polygon_area_xy(poly):
    pts = np.array(poly, dtype=np.float64).reshape(-1, 2)
    x, y = pts[:, 0], pts[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def largest_polygon(polygons):
    return max(polygons, key=polygon_area_xy)

def to_yolo_seg_line(class_id, polygon, w, h):
    pts = np.array(polygon, dtype=np.float64).reshape(-1, 2)
    if len(pts) < 3:
        return None
    pts[:, 0] = np.clip(pts[:, 0] / w, 0.0, 1.0)
    pts[:, 1] = np.clip(pts[:, 1] / h, 0.0, 1.0)
    coords = ' '.join(f'{v:.6f}' for v in pts.flatten())
    return f'{class_id} {coords}'


_n_converted_images = 0
_n_converted_instances = 0
_n_dropped_degenerate = 0

for src_path_str, rec in HITL_RECORDS.items():
    src_path = Path(src_path_str)

    w, h = rec.get('width'), rec.get('height')
    if not w or not h:
        img = cv2.imread(str(src_path))
        if img is None:
            continue
        h, w = img.shape[:2]

    lines = []
    for inst in rec['instances']:
        canon = map_to_canonical(inst['category'])
        if canon is None:
            continue  # unreachable given the Section 5 coverage assert, kept defensive
        poly = largest_polygon(inst['polygons'])
        line = to_yolo_seg_line(CANONICAL_INDEX[canon], poly, w, h)
        if line is None:
            _n_dropped_degenerate += 1
            continue
        lines.append(line)

    if not lines:
        continue

    # Stable, collision-free name across merged annotation files.
    safe_stem = hashlib.md5(src_path_str.encode('utf-8')).hexdigest()[:16]
    out_img_path = os.path.join(CONVERTED_IMAGES_DIR, f'{safe_stem}{src_path.suffix.lower()}')
    out_lbl_path = os.path.join(CONVERTED_LABELS_DIR, f'{safe_stem}.txt')

    if not os.path.exists(out_img_path):
        shutil.copy2(src_path, out_img_path)
    with open(out_lbl_path, 'w') as f:
        f.write('\n'.join(lines) + '\n')

    _n_converted_images += 1
    _n_converted_instances += len(lines)

print(f'Converted {_n_converted_images} images, {_n_converted_instances} instances '
      f'({_n_dropped_degenerate} degenerate polygons dropped).')


In [ ]:
# Verify the conversion by eye: redraw 5 written label files back onto their images.
_check_stems = random.Random(SEED).sample(
    [p.stem for p in Path(CONVERTED_IMAGES_DIR).glob('*') ], k=5
)
for stem in _check_stems:
    img_candidates = list(Path(CONVERTED_IMAGES_DIR).glob(f'{stem}.*'))
    if not img_candidates:
        continue
    img_path = img_candidates[0]
    label_path = Path(CONVERTED_LABELS_DIR) / f'{stem}.txt'
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    for line in label_path.read_text().strip().splitlines():
        parts = line.split()
        class_id = int(parts[0])
        coords = np.array(parts[1:], dtype=np.float64).reshape(-1, 2)
        coords[:, 0] *= w
        coords[:, 1] *= h
        pts = coords.astype(np.int32)
        cv2.polylines(img, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
        label_name = CANONICAL_CLASSES[class_id]
        cv2.putText(img, label_name, tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1, cv2.LINE_AA)
    out_path = os.path.join(EXP_DIR, 'label_overlay_check', f'{stem}.jpg')
    cv2.imwrite(out_path, img)
    print('Saved:', out_path)


## ✂️ Section 6 — Leak-safe train/val/test splits

Image-level split (never splits an image's instances across sets), fixed seed, with an
exact-duplicate check across splits (same MD5 approach used in exp03) before anything
is trained on. `val` drives early stopping; `test` is graded once at the end in
Section 8a.

In [ ]:
import random as _random

all_stems = sorted(p.stem for p in Path(CONVERTED_IMAGES_DIR).glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
rng = _random.Random(SEED)
shuffled = all_stems[:]
rng.shuffle(shuffled)

n = len(shuffled)
n_train = int(n * 0.8)
n_val = int(n * 0.1)
SPLIT_STEMS = {
    'train': shuffled[:n_train],
    'val': shuffled[n_train:n_train + n_val],
    'test': shuffled[n_train + n_val:],
}
for split, stems in SPLIT_STEMS.items():
    print(f'{split}: {len(stems)} images')


In [ ]:
# Physically materialize each split under DATASET_DIR/<split>/{images,labels}.
for split, stems in SPLIT_STEMS.items():
    img_out = os.path.join(DATASET_DIR, split, 'images')
    lbl_out = os.path.join(DATASET_DIR, split, 'labels')
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)
    for stem in stems:
        src_imgs = list(Path(CONVERTED_IMAGES_DIR).glob(f'{stem}.*'))
        if not src_imgs:
            continue
        src_img = src_imgs[0]
        src_lbl = os.path.join(CONVERTED_LABELS_DIR, f'{stem}.txt')
        dst_img = os.path.join(img_out, src_img.name)
        dst_lbl = os.path.join(lbl_out, f'{stem}.txt')
        if not os.path.exists(dst_img):
            shutil.copy2(src_img, dst_img)
        if not os.path.exists(dst_lbl):
            shutil.copy2(src_lbl, dst_lbl)

    manifest_path = os.path.join(DATASET_DIR, split, 'manifest.txt')
    with open(manifest_path, 'w') as f:
        f.write('\n'.join(stems) + '\n')

print('Splits materialized under', DATASET_DIR)


In [ ]:
# Exact-duplicate check across splits (same approach as exp03) — must run clean
# before training. Not a full near-duplicate/perceptual-hash search: HITL is a single
# real-photo source with its own (presumably already-disjoint) images, so exact-byte
# duplicates are the leak we're actually guarding against here.
def file_md5(path, chunk=65536):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

split_hashes = {}
for split in SPLIT_STEMS:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    split_hashes[split] = {file_md5(p): p.name for p in Path(img_dir).glob('*') if p.is_file()}

for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = set(split_hashes[a]) & set(split_hashes[b])
    if overlap:
        raise RuntimeError(f'{len(overlap)} byte-identical images appear in BOTH {a} and {b}.')

print('Leak check passed: no byte-identical images across train/val/test.')


In [ ]:
import yaml

DATA_YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')
data_yaml = {
    'path': DATASET_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {i: name for i, name in enumerate(CANONICAL_CLASSES)},
}
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

split_info = {
    'dataset_version': DATASET_VERSION,
    'source': 'humansintheloop/car-parts-and-car-damages (Kaggle, CC0)',
    'seed': SEED,
    'split_kind': 'fixed_seed_80_10_10_image_level',
    'counts': {split: len(stems) for split, stems in SPLIT_STEMS.items()},
    'canonical_classes': CANONICAL_CLASSES,
    'panel_classes': sorted(PANEL_CLASSES),
    'built_for_experiment': EXP_ID,
}
split_info_path = os.path.join(EXP_DIR, 'split_info.yaml')
with open(split_info_path, 'w') as f:
    yaml.dump(split_info, f, sort_keys=False)

print('Saved:', DATA_YAML_PATH)
print('Saved:', split_info_path)


## 🏋️ Section 7 — Train YOLO-seg (single run)

Pretrained seg checkpoint (`yolov8s-seg.pt`, generic COCO weights — see the Section 0
assumption on why this deviates from reusing exp01–03's `best.pt`), imgsz 640,
crop/zoom-biased augmentation to bridge HITL's typically wider framing toward
CrashLens' close-ups, early stopping on the HITL val split, fixed seed. `project=`
points at Drive with `save_period=1` so `last.pt` lands there every epoch, and the
cell is resume-aware — safe to re-run after any disconnect.

In [ ]:
from ultralytics import YOLO

MODEL_SOURCE = 'yolov8s-seg.pt'   # generic COCO-pretrained seg checkpoint; auto-downloaded by ultralytics
IMG_SIZE = 640
EPOCHS = 150
PATIENCE = 20          # early stopping on val, since HITL is the only signal we have here
WEIGHT_DECAY = 0.0005  # ultralytics default; HITL is a single, un-augmented-by-mixing dataset so no extra regularization bump
BATCH = 16             # lower if this OOMs on a T4 at imgsz 640

# Crop/zoom-biased augmentation: wider 'scale' range + translate exposes the model to
# more zoomed-in, off-center crops during training, bridging HITL's wider/full-car
# framing toward CrashLens' close-up captures. Left/right flip only (no vertical flip
# — not physically meaningful for a car). copy_paste/mixup off: risk of incoherent
# part boundaries when pasted across unrelated car photos.
AUGMENT = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.2, scale=0.8, shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0,
    mosaic=1.0, close_mosaic=10,
    mixup=0.0, copy_paste=0.0,
)

RUNS_PROJECT = os.path.join(EXP_DIR, 'runs')
RUN_NAME = EXP_ID
last_pt = os.path.join(RUNS_PROJECT, RUN_NAME, 'weights', 'last.pt')

if os.path.exists(last_pt):
    print(f'Found existing checkpoint at {last_pt} — resuming training.')
    model = YOLO(last_pt)
    try:
        train_results = model.train(resume=True)
    except Exception as e:
        if 'already' in str(e).lower():
            print('Training was already complete:', e)
            train_results = None
        else:
            raise
else:
    model = YOLO(MODEL_SOURCE)
    train_results = model.train(
        data=DATA_YAML_PATH,
        imgsz=IMG_SIZE,
        epochs=EPOCHS,
        patience=PATIENCE,
        seed=SEED,
        weight_decay=WEIGHT_DECAY,
        batch=BATCH,
        project=RUNS_PROJECT,
        name=RUN_NAME,
        exist_ok=True,
        save_period=1,   # last.pt lands on Drive every epoch — survives a disconnect mid-run
        **AUGMENT,
    )


In [ ]:
# Copy final artifacts into the flat EXP_DIR layout used by every other experiment.
run_dir = os.path.join(RUNS_PROJECT, RUN_NAME)
for fname in ('best.pt', 'last.pt'):
    src = os.path.join(run_dir, 'weights', fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(EXP_DIR, 'weights', fname))

for fname in ('results.csv', 'results.png'):
    src = os.path.join(run_dir, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(EXP_DIR, fname))

BEST_PT = os.path.join(EXP_DIR, 'weights', 'best.pt')
print('BEST_PT:', BEST_PT, '(exists)' if os.path.exists(BEST_PT) else '(MISSING — check the training cell above)')


## 📊 Section 8a — Evaluate on the HITL test split (automatic, has ground truth)

Proves the model learned HITL. Panel part-accuracy, per-class precision/recall/mAP50,
and a neighboring-parts confusion matrix restricted to `door` / `fender` / `bumper` /
etc. — the confusions that actually matter for costing.

In [ ]:
eval_model = YOLO(BEST_PT)
CONF_THRESH, IOU_THRESH = 0.25, 0.7

test_metrics = eval_model.val(
    data=DATA_YAML_PATH,
    split='test',
    imgsz=IMG_SIZE,
    conf=CONF_THRESH,
    iou=IOU_THRESH,
    project=EXP_DIR,
    name='hitl_test_eval',
    plots=True,
    exist_ok=True,
)
print('HITL test eval saved under', os.path.join(EXP_DIR, 'hitl_test_eval'))


In [ ]:
# Per-class precision/recall/mAP50, defensive against Ultralytics attribute drift
# across versions (same guard pattern used in exp03).
per_class_rows = []
try:
    seg = test_metrics.seg
    for i, cls_id in enumerate(seg.ap_class_index):
        cls_id = int(cls_id)
        per_class_rows.append({
            'class': CANONICAL_CLASSES[cls_id],
            'is_panel': CANONICAL_CLASSES[cls_id] in PANEL_CLASSES,
            'precision': float(seg.p[i]),
            'recall': float(seg.r[i]),
            'mAP50': float(seg.ap50[i]),
        })
except Exception as e:
    print('Could not extract per-class seg metrics (Ultralytics API drift?):', e)

per_class_df = pd.DataFrame(per_class_rows).sort_values('is_panel', ascending=False)
print(per_class_df.to_string(index=False))
per_class_csv = os.path.join(EXP_DIR, 'hitl_test_per_class_metrics.csv')
per_class_df.to_csv(per_class_csv, index=False)
print('Saved:', per_class_csv)


In [ ]:
# Neighboring-parts confusion matrix, restricted to panel classes (door/fender/bumper/...).
# ConfusionMatrix.matrix is (nc+1, nc+1): rows/cols 0..nc-1 are class predictions/labels
# in data.yaml order, the last row/col is background (missed / extra detections).
try:
    cm = test_metrics.confusion_matrix.matrix  # raw counts
    names = CANONICAL_CLASSES + ['background']
    cm_df = pd.DataFrame(cm, index=names, columns=names)
    cm_csv = os.path.join(EXP_DIR, 'confusion_matrix', 'hitl_test_confusion_matrix_full.csv')
    cm_df.to_csv(cm_csv)
    print('Saved full confusion matrix:', cm_csv)

    panel_names = [c for c in CANONICAL_CLASSES if c in PANEL_CLASSES]
    panel_cm_df = cm_df.loc[panel_names, panel_names]
    print('\nNeighboring-panel confusion (rows=predicted, cols=true):')
    print(panel_cm_df.to_string())
    panel_cm_csv = os.path.join(EXP_DIR, 'confusion_matrix', 'hitl_test_confusion_matrix_panels.csv')
    panel_cm_df.to_csv(panel_cm_csv)
    print('Saved:', panel_cm_csv)

    # Panel part-accuracy: true positives on the panel diagonal / all true panel instances
    # (row-sum over the panel+background columns for each panel's true-label column).
    panel_col_totals = cm_df.loc[:, panel_names].sum(axis=0)  # all predictions landing under each true panel col...
    true_panel_totals = cm_df.loc[panel_names, panel_names].sum(axis=0) + cm_df.loc['background', panel_names]
    panel_tp = pd.Series({p: cm_df.loc[p, p] for p in panel_names})
    panel_accuracy_pct = 100.0 * panel_tp.sum() / true_panel_totals.sum() if true_panel_totals.sum() else float('nan')
    print(f'\nHITL-test PANEL accuracy (headline metric): {panel_accuracy_pct:.1f}%  '
          f'({int(panel_tp.sum())}/{int(true_panel_totals.sum())} true panel instances correctly classified)')
except Exception as e:
    print('Could not build the panel confusion matrix (Ultralytics API drift?):', e)
    panel_accuracy_pct = float('nan')


## 📊 Section 8a-ii — split-pair detail (NEW in exp05, not in exp04)

Three cells, all reading `per_class_df`/`cm_df` already computed above (no re-inference):
1. **back_windshield** — explicit P/R/mAP + test-n, flagged low-confidence (n≈27).
2. **Combined-glass check** — windshield+back_windshield merged back together and
   compared to exp04's single `windshield` recall. This is the regression gate: if the
   split *hurt* overall glass detection, it shows up here.
3. **Split-pair summary** — fender↔quarter_panel, front_door↔back_door,
   front_wheel↔back_wheel, each half's own numbers next to exp04's merged-class recall.

In [ ]:
# back_windshield detail (NEW in exp05): the thinnest split half (271 raw HITL
# instances, ~27 test images) -- report its test-set numbers explicitly and flag
# low-n rather than silently blending it into the panel/non-panel averages above.
BACK_WINDSHIELD_RAW_INSTANCES = 271  # from exp04's hitl_instance_counts_raw.csv, pre-mapping
LOW_N_THRESHOLD = 30

bw_row = per_class_df[per_class_df['class'] == 'back_windshield']
bw_n_test = int(cm_df.loc[:, 'back_windshield'].sum()) if 'back_windshield' in cm_df.columns else None

if bw_row.empty:
    print('⚠️ back_windshield did not appear in the per-class seg metrics at all -- '
          'likely zero predictions. Treat as untested, not as "performed well".')
    bw_precision = bw_recall = bw_map50 = None
else:
    r = bw_row.iloc[0]
    bw_precision, bw_recall, bw_map50 = float(r['precision']), float(r['recall']), float(r['mAP50'])
    print(f"back_windshield -- P={bw_precision:.3f}  R={bw_recall:.3f}  mAP50={bw_map50:.3f}  "
          f"test instances={bw_n_test}")

bw_low_confidence = bw_n_test is None or bw_n_test < LOW_N_THRESHOLD
if bw_low_confidence:
    print(f"⚠️ LOW-CONFIDENCE: {bw_n_test} test instances (< {LOW_N_THRESHOLD}) -- do not "
          f"treat this number as a stable estimate of real-world back_windshield accuracy.")

back_windshield_report = pd.DataFrame([{
    'class': 'back_windshield',
    'raw_hitl_instances': BACK_WINDSHIELD_RAW_INSTANCES,
    'test_instances': bw_n_test,
    'precision': bw_precision,
    'recall': bw_recall,
    'mAP50': bw_map50,
    'low_confidence': bw_low_confidence,
}])
back_windshield_csv = os.path.join(EXP_DIR, 'back_windshield_report.csv')
back_windshield_report.to_csv(back_windshield_csv, index=False)
print('Saved:', back_windshield_csv)


In [ ]:
# Combined-glass check (NEW in exp05): treat windshield+back_windshield as ONE class
# (as exp04 did) and compare recall to exp04's own 'windshield' row -- the split must
# NOT reduce overall glass detection just by existing. Recomputed from this run's raw
# confusion matrix (not a re-trained merged model), so it's a fair "what if we'd kept
# them merged" read on these predictions.
windshield_cols = [c for c in ('windshield', 'back_windshield') if c in cm_df.columns]
windshield_rows = [c for c in ('windshield', 'back_windshield') if c in cm_df.index]

combined_true = int(cm_df.loc[:, windshield_cols].sum().sum())
combined_tp = int(cm_df.loc[windshield_rows, windshield_cols].sum().sum())  # either half predicted correctly counts
combined_recall = combined_tp / combined_true if combined_true else float('nan')

print(f'Combined windshield+back_windshield (exp05, this run): '
      f'recall={combined_recall:.3f}  (TP={combined_tp}/{combined_true} true instances)')

exp04_metrics_path = os.path.join(EXP04_DIR, 'hitl_test_per_class_metrics.csv')
if os.path.isfile(exp04_metrics_path):
    exp04_df = pd.read_csv(exp04_metrics_path)
    exp04_windshield = exp04_df[exp04_df['class'] == 'windshield']
    if not exp04_windshield.empty:
        exp04_recall = float(exp04_windshield.iloc[0]['recall'])
        delta = combined_recall - exp04_recall
        print(f"exp04 merged 'windshield' recall: {exp04_recall:.3f}  "
              f"(exp05 combined delta: {delta:+.3f})")
        if delta < -0.03:
            print('⚠️ Splitting windshield/back_windshield REGRESSED overall glass '
                  'detection vs. exp04 -- consider MERGE_REAR_WINDSHIELD=True at '
                  'inference (run_offline.py) instead of shipping the split.')
    else:
        print("exp04's per-class table has no 'windshield' row -- cannot compare.")
else:
    print(f'exp04 metrics not found at {exp04_metrics_path} -- skipping comparison '
          '(EXP04_DIR not reachable from this runtime).')

combined_glass_report = pd.DataFrame([{
    'windshield_combined_recall_exp05': combined_recall,
    'windshield_combined_true_instances_exp05': combined_true,
}])
combined_glass_report.to_csv(os.path.join(EXP_DIR, 'combined_glass_check.csv'), index=False)
print('Saved:', os.path.join(EXP_DIR, 'combined_glass_check.csv'))


In [ ]:
# Split-pair summary (NEW in exp05): for the other three pairs, report each half's own
# P/R/mAP (this run) next to exp04's single merged-class recall (reference only -- exp04
# never had to distinguish the halves, so this isn't a 1:1 comparison, but it surfaces
# whether either half collapsed to near-zero, which a single merged number could hide).
# windshield/back_windshield is covered separately by the combined-glass cell above.
exp04_metrics_path = os.path.join(EXP04_DIR, 'hitl_test_per_class_metrics.csv')
exp04_df = pd.read_csv(exp04_metrics_path) if os.path.isfile(exp04_metrics_path) else None
if exp04_df is None:
    print(f'exp04 metrics not found at {exp04_metrics_path} -- exp04 columns below will be empty.')

split_pair_rows = []
for merged_name, (half_a, half_b) in SPLIT_PAIRS.items():
    if merged_name == 'windshield':
        continue  # covered by the combined-glass cell above
    for half in (half_a, half_b):
        row = per_class_df[per_class_df['class'] == half]
        exp04_row = exp04_df[exp04_df['class'] == merged_name] if exp04_df is not None else None
        split_pair_rows.append({
            'pair': merged_name,
            'half': half,
            'exp05_precision': float(row.iloc[0]['precision']) if not row.empty else None,
            'exp05_recall': float(row.iloc[0]['recall']) if not row.empty else None,
            'exp05_mAP50': float(row.iloc[0]['mAP50']) if not row.empty else None,
            'exp05_test_instances': int(cm_df.loc[:, half].sum()) if half in cm_df.columns else None,
            'exp04_merged_class_recall': (
                float(exp04_row.iloc[0]['recall']) if exp04_row is not None and not exp04_row.empty else None
            ),
        })

split_pair_df = pd.DataFrame(split_pair_rows)
print(split_pair_df.to_string(index=False))
split_pair_csv = os.path.join(EXP_DIR, 'split_pair_vs_exp04.csv')
split_pair_df.to_csv(split_pair_csv, index=False)
print('Saved:', split_pair_csv)


## 🚗 Section 8b — Evaluate on the 20 real CrashLens close-ups (final reality check)

Only this part proves real transfer — Section 8a only proves the model learned HITL.
Builds a contact sheet + per-image overlays (reusing `save_contact_sheet` /
`draw_polygons`-equivalent code from Section 4), and computes **panel accuracy** against
a small, shared, manually-labeled ground-truth CSV kept on Drive (`REAL_GT_PATH`) — no
automated ground truth exists for these photos, same limitation exp01–03 documented.
If the GT file doesn't exist yet, this cell creates a template (predictions pre-filled
as a hint) and stops so you can fill in `true_panel` by eye before re-running.

In [ ]:
real_images = sorted(
    p for p in Path(REAL_CROPPED_DIR).glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')
)
print(f'{len(real_images)} real CrashLens close-ups found at {REAL_CROPPED_DIR}')

preds_dir = os.path.join(EXP_DIR, 'val_preds')
os.makedirs(preds_dir, exist_ok=True)

pred_records = []
contact_panels = []
for img_path in real_images:
    result = eval_model.predict(
        source=str(img_path), conf=CONF_THRESH, iou=IOU_THRESH, imgsz=IMG_SIZE, verbose=False,
    )[0]
    boxes = result.boxes
    num_dets = 0 if boxes is None else len(boxes)
    classes = [] if num_dets == 0 else [CANONICAL_CLASSES[int(c)] for c in boxes.cls]
    confs = [] if num_dets == 0 else [float(c) for c in boxes.conf]

    # top-1 among PANEL predictions only, since panels are what's judged/costed here
    panel_hits = [(c, cf) for c, cf in zip(classes, confs) if c in PANEL_CLASSES]
    top1_panel, top1_conf = (max(panel_hits, key=lambda t: t[1]) if panel_hits else (None, None))

    annotated = result.plot()
    out_path = os.path.join(preds_dir, f'{img_path.stem}_pred.jpg')
    cv2.imwrite(out_path, annotated)

    pred_records.append({
        'image': img_path.name,
        'num_detections': num_dets,
        'detected_classes': ';'.join(classes),
        'top1_panel_pred': top1_panel,
        'top1_panel_conf': top1_conf,
        'overlay_path': out_path,
    })
    contact_panels.append((cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), f'{img_path.name}\npred: {top1_panel or "-"}'))

pred_df = pd.DataFrame(pred_records)
pred_csv = os.path.join(EXP_DIR, 'real_eval_predictions.csv')
pred_df.to_csv(pred_csv, index=False)
print('Saved:', pred_csv)

real_contact_sheet_path = os.path.join(EXP_DIR, 'contact_sheet_real_closeups.png')
save_contact_sheet(contact_panels, real_contact_sheet_path, suptitle='Real CrashLens close-ups — exp04 predictions')
print('Saved:', real_contact_sheet_path)


In [ ]:
# Ground-truth compatibility check (NEW in exp05). REAL_GT_PATH's true_panel values were
# hand-labeled against exp04's 8 panel classes ('door', 'fender', ...). Under the new
# 10-class schema: 'door' and 'wheel' no longer exist as panel labels at all (they're now
# front_door/back_door, front_wheel/back_wheel -- 'wheel' was never a panel anyway), and
# any row still saying plain 'fender' is ambiguous (the annotator may not have
# distinguished it from quarter_panel). This does NOT block the run -- it just tells you,
# before scoring, which specific images need a quick manual re-check for a fair exp05
# number, instead of those rows silently dropping out of `judged` or silently comparing
# against a now-ambiguous label.
if os.path.exists(REAL_GT_PATH):
    _gt_check_df = pd.read_csv(REAL_GT_PATH)
    _stale_labels = {'door', 'wheel'}  # no longer valid panel labels at all under the split
    _ambiguous_labels = {'fender'}      # valid class name, but may predate the quarter_panel split

    _stale_rows = _gt_check_df[_gt_check_df['true_panel'].isin(_stale_labels)]
    _ambiguous_rows = _gt_check_df[_gt_check_df['true_panel'].isin(_ambiguous_labels)]

    if not _stale_rows.empty:
        print(f"⚠️ {len(_stale_rows)} row(s) in {REAL_GT_PATH} use a label that no longer "
              f"exists as a panel class ('door'/'wheel') -- these will be silently EXCLUDED "
              f"from exp05's panel-accuracy scoring below unless you re-label them as "
              f"front_door/back_door or front_wheel/back_wheel (or 'not_a_panel'/'unclear'):")
        print(_stale_rows[['image', 'true_panel']].to_string(index=False))
    if not _ambiguous_rows.empty:
        print(f"\n⚠️ {len(_ambiguous_rows)} row(s) are labeled plain 'fender' -- verify by eye "
              f"whether any of these are actually the rear quarter panel and should be "
              f"relabeled 'quarter_panel' for a fair exp05 number:")
        print(_ambiguous_rows[['image', 'true_panel']].to_string(index=False))
    if _stale_rows.empty and _ambiguous_rows.empty:
        print(f'{REAL_GT_PATH}: no stale or ambiguous labels found -- safe to score as-is.')
else:
    print(f'{REAL_GT_PATH} does not exist yet -- the next cell will create a template.')


In [ ]:
# Panel accuracy against a shared, manually-labeled ground-truth CSV. Only images whose
# true damaged part is one of the 8 panel classes are judged — non-panel real photos
# (e.g. a glass/lamp/wheel close-up) aren't this model's job to cost, so they're excluded.
if not os.path.exists(REAL_GT_PATH):
    template = pred_df[['image', 'top1_panel_pred']].copy()
    template = template.rename(columns={'top1_panel_pred': 'model_hint_ignore_me'})
    template['true_panel'] = ''  # fill by hand: one of PANEL_CLASSES, or 'not_a_panel' / 'unclear'
    template.to_csv(REAL_GT_PATH, index=False)
    print(f'No ground-truth file found — wrote a template to {REAL_GT_PATH}.')
    print('Open it, fill in true_panel for each image by eye, then re-run this cell.')
    real_panel_accuracy_pct = None
else:
    gt_df = pd.read_csv(REAL_GT_PATH)
    joined = pred_df.merge(gt_df[['image', 'true_panel']], on='image', how='left')
    judged = joined[joined['true_panel'].isin(PANEL_CLASSES)]
    if judged.empty:
        print(f'{REAL_GT_PATH} exists but has no filled-in panel labels yet — nothing to score.')
        real_panel_accuracy_pct = None
    else:
        correct = (judged['top1_panel_pred'] == judged['true_panel']).sum()
        real_panel_accuracy_pct = 100.0 * correct / len(judged)
        print(f'Real CrashLens PANEL accuracy: {real_panel_accuracy_pct:.1f}% '
              f'({correct}/{len(judged)} panel-labeled images)')

        exp03_reference_pct = 0.0  # as given in the exp04 brief — see Section 0 provenance flag
        print(f'\nexp03 (reported) -> exp04 (measured) real panel accuracy: '
              f'{exp03_reference_pct:.1f}% -> {real_panel_accuracy_pct:.1f}% '
              f'(delta {real_panel_accuracy_pct - exp03_reference_pct:+.1f} pts)')
        print('Caveat: the exp03 figure is carried forward from the task brief, not independently '
              'verified against exp03\'s own saved notebooks — see Section 0.')


## 🏷️ Section 8c — 31-photo Tagdeer set: association accuracy on the split labels (NEW in exp05)

Scores `panel_ground_truth_31photos.csv` (you attach this + the 31 images before running
this section — set `PANEL_GT_31PHOTOS_PATH` / `PHOTOS_31_DIR` in Section 0 if they land
somewhere other than the defaults). For every row flagged `tests_split=yes` (26 panels
per the task), this checks whether the damage region's **IoA≥0.40-associated part**
(mirroring `part_association_service.py`'s `ioa_box_mask` + `IOA_THRESHOLD=0.40` — the
exact rule, reimplemented inline below so this notebook stays self-contained in Colab,
not imported, since the backend package pulls in Firebase/FastAPI deps that don't belong
in a training notebook) resolves to the **correct split label** (e.g. `front_door` vs
`back_door`, not just "a door").

**Expected CSV schema (inferred — confirm/adjust the column names in the CONFIG cell
below if your actual file differs; this notebook does not silently guess a different
layout):**
- `image_id` — filename of one of the 31 photos (resolved against `PHOTOS_31_DIR`)
- `damage_index` — which damage region within that image (0-based), for images with more than one
- `bbox` — damage region box as `"x1,y1,x2,y2"` (pixel coords) — OR four separate
  `bbox_x1`/`bbox_y1`/`bbox_x2`/`bbox_y2` columns (auto-detected)
- `tests_split` — `yes`/`no`; only `yes` rows (expected 26) are scored
- `true_split_label` — the correct canonical label, one of the 8 split-pair halves
  (`fender`, `quarter_panel`, `front_door`, `back_door`, `front_wheel`, `back_wheel`,
  `windshield`, `back_windshield`)

Output: `association_accuracy_31photos.csv` (per-row predicted vs. true + hit/miss) and
a printed accuracy + misses list, both saved under `EXP_DIR`.

In [ ]:
# Inline mirror of part_association_service.py's ioa_box_mask + IOA_THRESHOLD (0.40) --
# see Section 8c's markdown for why this is reimplemented rather than imported.
ASSOC_IOA_THRESHOLD = 0.40
SPLIT_LABELS = {'fender', 'quarter_panel', 'front_door', 'back_door',
                 'front_wheel', 'back_wheel', 'windshield', 'back_windshield'}

def ioa_box_mask(box, mask):
    """Intersection of part mask with the damage box, over the box area -- identical
    formula to backend_api/services/part_association_service.py's ioa_box_mask."""
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1 = max(0, x1), max(0, y1)
    x2 = min(mask.shape[1], x2)
    y2 = min(mask.shape[0], y2)
    area = (x2 - x1) * (y2 - y1)
    if area <= 0:
        return 0.0
    return float(mask[y1:y2, x1:x2].sum()) / float(area)


def _parse_bbox(row):
    if 'bbox' in row and pd.notna(row['bbox']):
        parts = [float(v) for v in str(row['bbox']).replace(',', ' ').split()]
        if len(parts) == 4:
            return tuple(parts)
    cols = ('bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2')
    if all(c in row for c in cols) and all(pd.notna(row[c]) for c in cols):
        return tuple(float(row[c]) for c in cols)
    return None


if not os.path.exists(PANEL_GT_31PHOTOS_PATH):
    raise FileNotFoundError(
        f'{PANEL_GT_31PHOTOS_PATH} not found -- upload panel_ground_truth_31photos.csv '
        f'(and the 31 images under {PHOTOS_31_DIR}) before running this cell. Adjust the '
        f'paths in Section 0 if they land elsewhere.'
    )

gt31_df = pd.read_csv(PANEL_GT_31PHOTOS_PATH)
required_cols = {'image_id', 'tests_split', 'true_split_label'}
missing_cols = required_cols - set(gt31_df.columns)
if missing_cols:
    raise ValueError(
        f'{PANEL_GT_31PHOTOS_PATH} is missing expected column(s) {missing_cols}. '
        f'Found columns: {list(gt31_df.columns)}. Fix the CSV or adjust Section 8c\'s '
        f'markdown + this cell\'s column names -- do not guess a different layout silently.'
    )

scored_rows = gt31_df[gt31_df['tests_split'].astype(str).str.lower() == 'yes'].copy()
print(f'{len(gt31_df)} total rows, {len(scored_rows)} flagged tests_split=yes '
      f'(task expects 26).')

_image_cache = {}
results = []
for idx, row in scored_rows.iterrows():
    image_id = row['image_id']
    bbox = _parse_bbox(row)
    true_label = str(row['true_split_label']).strip()

    if bbox is None:
        results.append({'image_id': image_id, 'damage_index': row.get('damage_index'),
                         'true_split_label': true_label, 'predicted_label': None,
                         'best_ioa': None, 'hit': False, 'note': 'unparseable bbox'})
        continue

    if image_id not in _image_cache:
        img_candidates = list(Path(PHOTOS_31_DIR).glob(f'{Path(image_id).stem}.*'))
        _image_cache[image_id] = str(img_candidates[0]) if img_candidates else None
    img_path = _image_cache[image_id]

    if img_path is None:
        results.append({'image_id': image_id, 'damage_index': row.get('damage_index'),
                         'true_split_label': true_label, 'predicted_label': None,
                         'best_ioa': None, 'hit': False, 'note': f'image not found under {PHOTOS_31_DIR}'})
        continue

    pred = eval_model.predict(source=img_path, conf=CONF_THRESH, iou=IOU_THRESH, imgsz=IMG_SIZE, verbose=False)[0]
    best_label, best_ioa = None, 0.0
    if pred.masks is not None:
        for i in range(len(pred.boxes)):
            label = CANONICAL_CLASSES[int(pred.boxes.cls[i])]
            if label not in SPLIT_LABELS:
                continue
            mask = pred.masks.data[i].cpu().numpy().astype(bool)
            if mask.shape != pred.orig_shape:
                mask = cv2.resize(mask.astype('uint8'), (pred.orig_shape[1], pred.orig_shape[0])).astype(bool)
            ioa = ioa_box_mask(bbox, mask)
            if ioa >= ASSOC_IOA_THRESHOLD and ioa > best_ioa:
                best_label, best_ioa = label, ioa

    hit = best_label == true_label
    results.append({'image_id': image_id, 'damage_index': row.get('damage_index'),
                     'true_split_label': true_label, 'predicted_label': best_label,
                     'best_ioa': round(best_ioa, 3) if best_label else None, 'hit': hit, 'note': ''})

assoc_df = pd.DataFrame(results)
scored = assoc_df[assoc_df['note'] == '']
accuracy = scored['hit'].mean() if len(scored) else float('nan')
print(f'\nAssociation accuracy on split labels: {accuracy:.1%}  ({int(scored["hit"].sum())}/{len(scored)} scored rows)')

misses = assoc_df[(assoc_df['hit'] == False)]
if not misses.empty:
    print(f'\n{len(misses)} miss(es):')
    print(misses.to_string(index=False))

assoc_csv = os.path.join(EXP_DIR, 'association_accuracy_31photos.csv')
assoc_df.to_csv(assoc_csv, index=False)
print('\nSaved:', assoc_csv)


## 📝 Section 9 — notes.md (docs §8 template)

In [ ]:
from datetime import date

hitl_counts_summary = counts_df.to_string(index=False)
split_counts_summary = '\n'.join(f'  {s}: {len(stems)} images' for s, stems in SPLIT_STEMS.items())

try:
    hitl_test_line = f'PANEL accuracy on HITL test split: {panel_accuracy_pct:.1f}%'
except NameError:
    hitl_test_line = 'PANEL accuracy on HITL test split: (fill in after running Section 8a)'

try:
    if real_panel_accuracy_pct is not None:
        real_eval_line = f'Real CrashLens PANEL accuracy: {real_panel_accuracy_pct:.1f}%'
    else:
        real_eval_line = 'Real CrashLens PANEL accuracy: (fill in — ground-truth CSV not yet labeled, see Section 8b)'
except NameError:
    real_eval_line = 'Real CrashLens PANEL accuracy: (fill in after running Section 8b)'

try:
    bw_line = (f"back_windshield: P={back_windshield_report.iloc[0]['precision']}  "
               f"R={back_windshield_report.iloc[0]['recall']}  "
               f"test_n={back_windshield_report.iloc[0]['test_instances']}  "
               f"low_confidence={back_windshield_report.iloc[0]['low_confidence']}")
except NameError:
    bw_line = '(fill in after running Section 8a-ii)'

try:
    glass_line = (f"combined windshield+back_windshield recall={combined_recall:.3f} "
                  f"vs exp04 merged windshield recall "
                  f"(see {os.path.join(EXP_DIR, 'combined_glass_check.csv')})")
except NameError:
    glass_line = '(fill in after running Section 8a-ii)'

try:
    assoc_line = f'31-photo association accuracy on split labels: {accuracy:.1%} ({len(scored)} rows scored)'
except NameError:
    assoc_line = '(fill in after running Section 8c, once panel_ground_truth_31photos.csv is attached)'

notes = f"""# {EXP_ID} — HITL part-panel segmentation, split classes
Date: {date.today().isoformat()}
Purpose (one variable under test): same dataset + recipe as exp04_hitl_panels -- the ONLY
  change is the class mapping (Section 5): four pairs exp04 merged are now kept separate
  (fender/quarter_panel, front_door/back_door, front_wheel/back_wheel,
  windshield/back_windshield). exp04 itself and its Drive outputs are untouched; this
  notebook only reads exp04's saved CSVs for side-by-side comparison.
Dataset version: {DATASET_VERSION} (source: humansintheloop/car-parts-and-car-damages, Kaggle, CC0)

Class mapping (21 HITL classes -> 20 canonical, 10 costed panels + 10 non-panel trained-only):
{CLASS_MAP_LOG}

HITL raw instance counts (pre-mapping):
{hitl_counts_summary}

Splits (seed={SEED}, image-level, fixed 80/10/10, leak check passed):
{split_counts_summary}

Config (lr, epochs, backbone frozen?, imgsz, augment) -- IDENTICAL to exp04 except the
  class mapping above; this is a one-variable experiment:
  model_source={MODEL_SOURCE} (generic COCO-pretrained, not the exp01-03 lineage checkpoint)
  imgsz={IMG_SIZE}, epochs={EPOCHS}, patience={PATIENCE}, weight_decay={WEIGHT_DECAY}, batch={BATCH}, seed={SEED}
  augment={AUGMENT}
  backbone frozen: no (single-phase, unfrozen throughout)

Training metrics: (fill in from {os.path.join(EXP_DIR, 'results.csv')} after reviewing results.png)

Validation metrics (independent set -- HITL test split):
  {hitl_test_line}
  Full per-class table: {os.path.join(EXP_DIR, 'hitl_test_per_class_metrics.csv')}
  Panel confusion matrix: {os.path.join(EXP_DIR, 'confusion_matrix', 'hitl_test_confusion_matrix_panels.csv')}

Split-pair detail (NEW in exp05, Section 8a-ii):
  {bw_line}
  {glass_line}
  Full split-pair-vs-exp04 table: {os.path.join(EXP_DIR, 'split_pair_vs_exp04.csv')}

Real-domain reality check (20 CrashLens close-ups, Section 8b):
  {real_eval_line}
  NOTE: REAL_GT_PATH was labeled against exp04's 8 panel classes -- Section 8b's new
  compatibility-check cell lists any stale ('door'/'wheel') or ambiguous ('fender') rows
  that needed a manual re-check before this number is comparable to exp04's.
  Contact sheet: {os.path.join(EXP_DIR, 'contact_sheet_real_closeups.png')}
  Predictions: {os.path.join(EXP_DIR, 'real_eval_predictions.csv')}
  Ground truth (shared, cross-experiment): {REAL_GT_PATH}

31-photo Tagdeer association accuracy on split labels (NEW in exp05, Section 8c):
  {assoc_line}
  Per-row results + misses list: {os.path.join(EXP_DIR, 'association_accuracy_31photos.csv')}

Visual predictions (paths):
  HITL label-conversion check: {os.path.join(EXP_DIR, 'label_overlay_check')}
  HITL raw contact sheet: {os.path.join(EXP_DIR, 'contact_sheet_hitl_raw.png')}
  Real close-up overlays: {os.path.join(EXP_DIR, 'val_preds')}

Strengths: (fill in after reviewing val_preds/ overlays and the panel confusion matrix)
Weaknesses: (fill in -- in particular whether back_windshield's low-n numbers above are
  usable at all, and whether MERGE_REAR_WINDSHIELD should stay True in run_offline.py)
Data/leakage checks done: image-level split, fixed seed, exact-duplicate (MD5) check across
  train/val/test -- passed. No cross-dataset leak risk (single dataset, no mixing).
Lessons learned: (fill in -- did any of the four splits collapse vs. its exp04 merged
  baseline per the combined-glass/split-pair cells? did the 31-photo association accuracy
  hold for the new split labels specifically, not just the HITL test split?)
Recommended next step: (fill in -- e.g. ship fender/door/wheel splits, hold back_windshield
  behind MERGE_REAR_WINDSHIELD pending more real back-windshield data; or targeted
  augmentation/annotation for whichever class came out weakest)
"""

notes_path = os.path.join(EXP_DIR, 'notes.md')
with open(notes_path, 'w', encoding='utf-8') as f:
    f.write(notes)
print('Saved:', notes_path)
print('⚠️ notes.md has placeholders -- fill in Strengths/Weaknesses/Lessons/Next-step '
      'after reviewing val_preds/ overlays before calling this experiment finished '
      '(per docs/segmentation_finetuning.md §7).')


In [ ]:
config = {
    'experiment': EXP_ID,
    'dataset_version': DATASET_VERSION,
    'dataset_source': 'humansintheloop/car-parts-and-car-damages (Kaggle, CC0)',
    'model_source': MODEL_SOURCE,
    'imgsz': IMG_SIZE,
    'epochs': EPOCHS,
    'patience': PATIENCE,
    'weight_decay': WEIGHT_DECAY,
    'batch': BATCH,
    'seed': SEED,
    'augment': AUGMENT,
    'canonical_classes': CANONICAL_CLASSES,
    'panel_classes': sorted(PANEL_CLASSES),
    'conf_thresh': CONF_THRESH,
    'iou_thresh': IOU_THRESH,
}
config_path = os.path.join(EXP_DIR, 'config.yaml')
with open(config_path, 'w') as f:
    yaml.dump(config, f, sort_keys=False)
print('Saved:', config_path)


## ✅ Summary

- Dataset: HITL alone, `v4_hitl_split`, 20 canonical classes (10 costed panels, up from
  exp04's 8; 10 non-panel, up from exp04's 8). Only change from exp04: the four split pairs.
- Weights: `EXP_DIR/weights/best.pt`, `last.pt`.
- HITL test metrics: `EXP_DIR/hitl_test_per_class_metrics.csv`, `EXP_DIR/confusion_matrix/`.
- Split-pair detail (NEW): `EXP_DIR/back_windshield_report.csv`, `EXP_DIR/combined_glass_check.csv`, `EXP_DIR/split_pair_vs_exp04.csv`.
- Real CrashLens reality check: `EXP_DIR/real_eval_predictions.csv`, `EXP_DIR/contact_sheet_real_closeups.png`.
- 31-photo association accuracy on split labels (NEW): `EXP_DIR/association_accuracy_31photos.csv`.
- Record: `EXP_DIR/notes.md`, `EXP_DIR/config.yaml`, `EXP_DIR/split_info.yaml`.

Headline metrics: panel accuracy (8a) and the three new split-pair checks (8a-ii) together
decide whether to ship all four splits, some of them, or fall back to
`MERGE_REAR_WINDSHIELD=True` for back_windshield specifically — not panel accuracy alone,
since a single blended number can hide one half of a pair collapsing. exp04 and its Drive
outputs were not modified by this notebook.